# Session 2 — ISS ground track

**Phase 0.** Propagate the ISS over ~90 minutes and plot its ground track — the classic sine wave.

*Concept:* pass the **whole time array** to `.at()` at once (vectorized) instead of looping. The track's north/south extent equals the orbital **inclination** (~51.6° for the ISS). Longitude wraps at ±180°, so expect a horizontal 'jump' — that's cosmetic, not a bug.

In [1]:
import numpy as np
from skyfield.api import load

ts = load.timescale()
url = "https://celestrak.org/NORAD/elements/gp.php?GROUP=stations&FORMAT=tle"
iss = next(s for s in load.tle_file(url) if "ISS" in s.name)

# 90 one-minute steps from now (vectorized).
import datetime as dt
now = dt.datetime.utcnow()
times = ts.utc(now.year, now.month, now.day, now.hour, now.minute + np.arange(90))
sp = iss.at(times).subpoint()
lat, lon = sp.latitude.degrees, sp.longitude.degrees
print(f"inclination-limited latitude range: {lat.min():.1f}\u00b0 .. {lat.max():.1f}\u00b0")

inclination-limited latitude range: -51.8° .. 51.8°


In [2]:
import plotly.graph_objects as go

fig = go.Figure(go.Scattergeo(lon=lon, lat=lat, mode="lines+markers",
                              line=dict(width=2, color="#4cc9f0"),
                              marker=dict(size=3)))
fig.update_geos(projection_type="natural earth", showland=True,
                landcolor="#1b2942", oceancolor="#0b1220", showocean=True)
fig.update_layout(title="ISS ground track (next 90 min)", height=500,
                  paper_bgcolor="#060a14", font_color="#e8eef7")
fig.write_html("iss_ground_track.html")
fig.show()

**Done when:** the track looks like the classic sine wave, peaking near ±51.6° latitude. Saved to `iss_ground_track.html`.